# 2부. 데이터 분석과 탐색

장별 완성 예제. 학습용 가상 데이터이며 공식 시험 제출 형식이 아니다. examples 폴더에서 전체 실행한다.

In [ ]:
from pathlib import Path
import os
if not Path('lab.py').is_file() and Path('../lab.py').is_file():
    os.chdir('..')
assert Path('lab.py').is_file(), 'Run with the examples folder and data present'
get_ipython().run_line_magic('matplotlib', 'inline')


## 05장. pandas로 데이터 불러오고 선택하기

CSV·DataFrame·행과 열 선택·조건 필터링을 수행한다.

In [ ]:
from lab import load_customers

df = load_customers()
columns = ["customer_id", "tenure_months", "monthly_fee"]
selected = df.loc[df["tenure_months"] >= 12, columns]
first = df.iloc[:3, :2]
single = df[["monthly_fee"]]
assert single.shape == (600, 1)
assert selected["tenure_months"].ge(12).all()
assert list(first.columns) == ["customer_id", "tenure_months"]
print(selected.head())

## 06장. 데이터 집계와 결합하기

그룹별 집계·병합·파생변수의 단위를 확인한다.

In [ ]:
import pandas as pd
from lab import ROOT, load_customers

df = load_customers()
summary = df.groupby("contract_type", dropna=False).agg(
    customers=("customer_id", "size"),
    mean_fee=("monthly_fee", "mean"),
    churn_rate=("churn", "mean"),
)
lookup = pd.read_csv(ROOT / "data/regions.csv")
joined = df.merge(lookup, on="region", how="left", validate="many_to_one")
assert len(joined) == len(df)
assert joined["zone"].notna().all()
ranked = df.sort_values("tenure_months", ascending=False)
assert ranked["tenure_months"].is_monotonic_decreasing
print(summary)

## 07장. 데이터 구성과 품질 점검하기

자료형·기술통계·결측·중복을 점검한다.

In [ ]:
from lab import load_customers

df = load_customers()
print(df.shape)
print(df.dtypes)
print(df[["tenure_months", "monthly_fee", "support_calls"]].describe())
missing = df.isna().sum().sort_values(ascending=False)
duplicate_ids = df["customer_id"].duplicated().sum()
print(missing, "duplicate IDs:", duplicate_ids)
assert duplicate_ids == 0
assert set(df["churn"].unique()) == {0, 1}
assert (df["monthly_fee"] < 0).sum() == 1

## 08장. 시각화로 데이터 특성 파악하기

분포·범주·변수 관계를 목적에 맞는 그림으로 확인한다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from lab import load_customers

df = load_customers()
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
sns.histplot(data=df, x="tenure_months", bins=12, ax=axes[0])
sns.countplot(data=df, x="contract_type", ax=axes[1])
axes[0].set_title("Tenure distribution")
axes[1].set_title("Contract counts")
fig.tight_layout()
assert len(axes) == 2
plt.show()
plt.close(fig)

## 09장. 상관관계와 타깃 분석하기

상관과 타깃 분포를 해석하고 누수를 점검한다.

In [ ]:
import pandas as pd
from lab import split_task

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
train = X_train.copy()
train["churn"] = y_train
numeric = train.select_dtypes(include="number")
correlations = numeric.corr()["churn"].drop("churn").sort_values()
counts = y_train.value_counts().sort_index()
rates = y_train.value_counts(normalize=True).sort_index()
assert len(X_train) == len(y_train) == 360
assert abs(rates.sum()-1) < 1e-10
print(counts, rates, correlations)